# Import libraries

In [1]:
import uuid
import copy
from datetime import datetime
from pathlib import Path
import getpass
import sys
import pandas as pd
import numpy as np
import yaml
import pickle
import re
import os
import glob
from typing import Dict, List, Optional, Any, Tuple
from tqdm.notebook import tqdm
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat

pd.options.display.max_colwidth = 300

In [ ]:
# Required: QC (model) and Target (reference) atlas for performing RT alignment
QC_ATLAS_UID = "atlas-c94e57c00d8245ebb72eef59c2862c8f"
TARGET_ATLAS_UID = "atlas-e1c4aea2f54f447dbb37531b6dabbe6f"

# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"

In [ ]:
# Load configuration from YAML file
config_path = Path("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

PROJECT_FILES_PATH = os.path.join(config["paths"]["projects_dir"], PROJECT_NAME, "lcmsruns")
METATLAS_DB_PATH = config["paths"]["main_database"]
PROJECT_DB_PATH = os.path.join(config["paths"]["projects_dir"], PROJECT_NAME, f"{PROJECT_NAME}.duckdb")

# Organize settings
METADATA = {
    "timestamp": datetime.now().isoformat(),
    "analyst": getpass.getuser()
}
DATA_EXTRACT_SETTINGS = {
    "default_ppm_tolerance": config["mass_tolerance"]["qc_extraction_ppm"],
    "window_expansion": config["rt_alignment"]["rt_window_expansion"],
    "min_peak_intensity": config["analysis_settings"]["min_peak_intensity"],
}
RT_SETTINGS = {
    "polynomial_degree": config["rt_alignment"]["polynomial_degree"],
    "min_observations_per_compound": config["rt_alignment"]["min_observations_per_compound"],
    "min_compounds_for_modeling": config["rt_alignment"]["min_compounds_for_modeling"],
    "exclude_inchikeys": config["rt_alignment"]["exclude_inchikeys"],
    "r2_threshold": config["rt_alignment"]["r2_threshold"],
    "max_residual_rt": config["rt_alignment"]["max_residual_rt"]
}

# Create Project Database

In [4]:
# Create project-specific database
dbi.create_project_database(PROJECT_DB_PATH, overwrite_existing=True)

Overwrite is set to True; Deleted existing database at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb and proceeding...
Project database created at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb


# Load or save LCMS runs

In [5]:
# Save LCMS runs to project database
lcmsrun_files = dbi.save_lcmsruns_to_db(
    project_db_path=PROJECT_DB_PATH,
    project_name=PROJECT_NAME,
    project_path=PROJECT_FILES_PATH,
    project_metadata=METADATA,
    overwrite_existing=True
)

Saved 8 LCMS runs to database:

Chromatography: HILICZ
  Polarity: FPS
    experimental: 0 files
    qc: 2 files
    injbl: 1 files
    exctrl: 0 files
    istd: 1 files
  Polarity: POS
    experimental: 3 files
    qc: 0 files
    injbl: 0 files
    exctrl: 1 files
    istd: 0 files


# Extract and Match QC Compounds

In [ ]:
matches_df, matching_stats = msa.extract_and_match_qc_compounds(
    project_db_path=PROJECT_DB_PATH,
    database_path=METATLAS_DB_PATH,
    qc_atlas_uid=QC_ATLAS_UID,
    metadata=DATA_EXTRACT_SETTINGS
)

Loading QC files and atlas compounds from databases...
Retrieved 20 compounds from main database for atlas: HILIC Positive QC Default
Atlas chromatography: HILICZ, polarity: positive
Found 2 QC files and 20 QC compounds
Extracting MS1 data from QC files...


Extracting MS1 data:   0%|          | 0/2 [00:00<?, ?it/s]

  Extracted 44810 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_pos)
  Extracted 19240 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_neg)
  Extracted 42533 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_pos)
  Extracted 19417 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_neg)
Total MS1 peaks extracted: 126,000

Matching QC compounds to extracted peaks...


Matching compounds:   0%|          | 0/20 [00:00<?, ?it/s]


Matching completed:
  Compounds with matches: 20/20
  Total compound-file matches: 40
  Mean m/z error: -0.21 ± 0.51 ppm
  Mean RT difference: 0.028 ± 0.119 min


# Build RT Alignment Model

In [7]:
# Build RT alignment model using QC compound matches
best_model, modeling_results_df, compound_rt_stats = rat.build_rt_alignment_model(matches_df, RT_SETTINGS)

Building RT alignment model...

RT Statistics Summary:
  Atlas RT range: 1.09 - 17.01 min
  Observed RT range (median): 1.25 - 17.14 min
  Mean RT difference (observed - atlas): 0.028 ± 0.120 min
Using 20 compounds with ≥1 observations (QC files) for modeling
Building polynomial model (degree 2)...

Model built successfully:
  Model type: Polynomial degree 2
  R² = 0.9996
  RMSE = 0.1041 min
  Max residual = 0.2801 min
  Equation: RT_corrected = 0.184345 + 0.956634 * RT_atlas + 0.002163 * RT_atlas^2

Compound RT Statistics:


,compound_name,inchi_key,ref_rt_peak,exp_rt_median,rt_diff_median,observation_count,exp_rt_std
0,Compound_0,LCMZECCEEOQWLQ-UHFFFAOYSA-N,1.0938,1.253400,0.1596,2,0.0001
1,Compound_13,AYFVYJQAPQTCCC-KJQIOZMZSA-N,13.4896,13.384000,-0.1055,2,0.0094
2,Compound_6,OVRNDRQMDRJTHS-ZEUBEQSHSA-N,6.9440,7.211600,0.2677,2,0.0238
3,Compound_12,QNAYBMKLOCPYGJ-UVYXLFMMSA-N,13.4051,13.358000,-0.0471,2,0.0095
4,Compound_16,CKLJMWTZIZZHCS-NYTNKSBQSA-N,16.1304,16.169500,0.0391,2,0.0116
5,Compound_9,FFEARJCKVFRZRR-XAFSXMPTSA-N,10.4410,10.409400,-0.0315,2,0.0075
6,Compound_2,GFFGJBXGBJISGV-CIKZIQIKSA-N,2.6776,2.816400,0.1388,2,0.0213
7,Compound_7,COLNVLDHVKWLRT-CMLFETTRSA-N,8.9793,9.131300,0.1520,2,0.0028
8,Compound_4,UGQMRVRMYYASKQ-SHKQESLVSA-N,5.4342,5.197600,-0.2366,2,0.0412
9,Compound_8,QIVBCDIJIAJPQS-HNEHNLBWSA-N,10.1566,10.180700,0.0240,2,0.0109


# Create RT-Corrected Experimental Data and Save to Database

In [ ]:
rt_summary = rat.apply_rt_correction_to_target(
                PROJECT_DB_PATH,
                METATLAS_DB_PATH,
                TARGET_ATLAS_UID,
                best_model,
                lcmsrun_files,
                modeling_results_df,
                METADATA
            )

Cloning target atlas and applying RT correction...
Retrieved atlas metadata for: HILIC Positive ISTD Default
Retrieved 65 compounds from main database for atlas: HILIC Positive ISTD Default
Atlas chromatography: HILICZ, polarity: positive
RT alignment model saved to database with UID: rta-c5a9c002c5224ceeb38e5ac960fbc5c5
Created RT-corrected atlas with 65 associations.
RT correction completed: 65/65 compounds corrected
Correction statistics: mean = 0.0207, std = 0.0534 min

RT-corrected atlas created:
  Atlas UID: atlas-rt-e4cada2bc6eb4006b49bbc4f43f4ed78
  Atlas name: HILIC Positive ISTD Default (RT Corrected)
  RT alignment model UID: rta-c5a9c002c5224ceeb38e5ac960fbc5c5
  Project database: /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb


In [9]:
atlas_info = dbi.list_available_atlases(str(PROJECT_DB_PATH))
if not atlas_info.empty:
    print(f"   Available atlases:")
    for _, row in atlas_info.iterrows():
        print(f"      {row['atlas_uid']}")
        print(f"            {row['atlas_name']}")
        print(f"            {row['chromatography']} {row['polarity']}")
        print(f"            {row['compound_count']} compounds")
        print(f"            {row['last_modified']}")
else:
    print(f"\n   No atlases found")

   Available atlases:
      atlas-rt-e4cada2bc6eb4006b49bbc4f43f4ed78
            HILIC Positive ISTD Default (RT Corrected)
            HILICZ positive
            65 compounds
            2025-08-27 16:37:05.089223


In [10]:
dbi.get_rt_correction_table_entry(PROJECT_DB_PATH, row['atlas_uid'])

,rt_alignment_uid,project_name,atlas_uid,model_type,polynomial_degree,r_squared,rmse,coefficients,equation,qc_files_count,compounds_used_count,created_by,last_modified,model_metadata
0,rta-c5a9c002c5224ceeb38e5ac960fbc5c5,20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519,atlas-rt-e4cada2bc6eb4006b49bbc4f43f4ed78,polynomial,2,0.999616,0.104061,"[0.0, 0.9566335548110365, 0.0021634959713895553]",RT_corrected = 0.184345 + 0.956634 * RT_atlas + 0.002163 * RT_atlas^2,2,20,BKieft,2025-08-27 16:37:05.089223,"{""qc_files"": [""20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5"", ""20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5""], ""compounds..."
